# XGBoost PU Bagging — feh-only 重要性采样（消融：是否只是 [Fe/H] 的影响）

**实验目的：** 上一版 3D（teff/logg/feh）重要性采样显著拉低了测试排序性能、放大了候选数。本实验做**消融**——只把 **[Fe/H]** 一个变量用于匹配/加权，检验"是否只是 feh 的影响"：
- 若 feh-only 几乎复现 3D 的效果 → [Fe/H] 是主导混淆变量，teff/logg 贡献很小
- 若 feh-only 明显更弱 → teff/logg 也贡献了不可忽略的混淆

**方法（与 3D 版完全同构，只是维度从 3 降到 1）：**
1. **feh 分箱匹配近邻**：一维 [Fe/H] 分位分箱（5 箱），每个正样本在同箱内取 K=10 个最近 feh 邻居
2. **feh KDE 加权**：一维 [Fe/H] 高斯核密度比 w = f_pos / f_unl 加权采样

**对照：** 标准 PU（等概率采样）在同划分、同随机种子下重跑；并与已保存的 3D 结果合并成五路对比。

In [ ]:
# 共享数据加载 + 导入重要性采样模块

import sys, os, time
from pathlib import Path

# 定位项目根目录（向上查找直到同时存在 ML/ 与 Data/）
_PROJECT_ROOT = Path(os.getcwd())
for _ in range(5):
    if (_PROJECT_ROOT / "ML").exists() and (_PROJECT_ROOT / "Data").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent
for p in (str(_PROJECT_ROOT), str(_PROJECT_ROOT / "XGB")):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 10})
import warnings
warnings.filterwarnings("ignore")

# 核心逻辑（采样器 + PU Bagging + 评估）统一在 XGB/importance_sampling.py
import importance_sampling as imp

t0 = time.time()
S = imp.load_and_split()
X_clean = S["X_clean"]
stars_clean = S["stars_clean"]
common_wave = S["common_wave"]
X_all = S["X_spec_scaled"]
X_tr = S["X_tr"]
X_te = S["X_te"]
y_tr = S["y_tr"]
y_te = S["y_te"]
n_pos_tr = S["n_pos_tr"]
unl_tr_idx = S["unl_tr_idx"]
tr_idx = S["tr_idx"]

feh_all = stars_clean["feh"].values.astype(float)

print(f"数据加载完成 ({time.time()-t0:.0f}s)")
print(f"总样本 {len(S['y_all']):,} | 训练 {len(tr_idx):,} (P={n_pos_tr}, U={len(unl_tr_idx):,}) | 测试 {len(S['test_idx']):,}")
print(f"物理参数列: teff/logg/feh（全样本无缺失）")


## 1. 构造 feh-only 采样器

只把 [Fe/H] 一个变量喂给匹配/KDE 构造，其余物理参数不参与。

In [ ]:
# feh-only 标准化 + 构造两个 feh-only 采样器

name_fm = "feh_matched"
name_fk = "feh_kde"
B, K = 5, 10

# 只用 [Fe/H] 一列做匹配 / 加权（cols=["feh"]），其余物理参数不参与
phys1, mu1, sd1 = imp.standardize_physics(stars_clean, cols=["feh"])
phys1_tr = phys1[tr_idx]
phys1_pos = phys1_tr[np.where(y_tr == 1)[0]]   # (n_pos_tr, 1)
phys1_unl = phys1_tr[unl_tr_idx]               # (n_unl, 1)

# ① feh-only 分箱匹配近邻（一维分箱）
cands_f, n_cell_f = imp.build_match_candidates(phys1_pos, phys1_unl, unl_tr_idx, B=B, K=K)
fm_sampler = imp.MatchSampler(cands_f, imp.random_seed)
feh_match_pool = np.unique(np.concatenate([np.atleast_1d(c) for c in cands_f]))

# ② feh-only KDE 加权（一维核密度）
w_f, ess_f = imp.build_kde_weights(phys1_pos, phys1_unl, unl_tr_idx)
fk_sampler = imp.WeightedSampler(unl_tr_idx, w_f, imp.random_seed)

print(f"feh-only 标准化: mu={mu1[0]:.3f}, sd={sd1[0]:.3f}")
print(f"① feh 分箱匹配: 分箱 B={B}, 近邻 K={K}, 分层命中 {n_cell_f}/{len(cands_f)}, "
      f"有效负池 {len(feh_match_pool):,}")
print(f"② feh KDE 加权: max/median={np.max(w_f)/np.median(w_f):.1f}, ESS={ess_f:.0f}/{len(unl_tr_idx):,}")


## 2. 采样分布诊断（[Fe/H]）

观察 feh-only 匹配/KDE 是否把负样本的 [Fe/H] 分布拉向正样本。

In [ ]:
# 诊断：feh-only 采样后负样本的 [Fe/H] 分布

feh_pos = feh_all[tr_idx[np.where(y_tr == 1)[0]]]
feh_unl = feh_all[tr_idx[unl_tr_idx]]
feh_fmatch = feh_all[tr_idx[feh_match_pool]]

bins = np.linspace(-2.6, -0.6, 41)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hist(feh_unl, bins=bins, density=True, alpha=0.45, color="steelblue",
        label=f"U 池（等概率, n={len(feh_unl):,}）")
ax.hist(feh_fmatch, bins=bins, density=True, alpha=0.6, color="darkorange",
        label=f"feh 分箱匹配负池（n={len(feh_fmatch):,}）")
ax.hist(feh_unl, bins=bins, weights=w_f, density=True, alpha=0.6, color="green",
        label="feh KDE 加权（有效分布）")
ax.hist(feh_pos, bins=bins, density=True, alpha=0.8, color="crimson",
        label=f"已知 CN 正样本（n={len(feh_pos)}）")
ax.axvline(-1.2, color="k", linestyle="--", linewidth=1.0, label="[Fe/H] = -1.2")
ax.set_xlabel("[Fe/H]"); ax.set_ylabel("密度")
ax.set_title("feh-only 重要性采样：负样本 [Fe/H] 分布对齐情况")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"U 池(等概率)     中位 feh={np.median(feh_unl):.3f}  feh>-1.2 占比={np.mean(feh_unl > -1.2)*100:.1f}%")
print(f"feh 分箱匹配负池 中位 feh={np.median(feh_fmatch):.3f}  feh>-1.2 占比={np.mean(feh_fmatch > -1.2)*100:.1f}%")
print(f"feh KDE 加权     中位 feh={np.average(feh_unl, weights=w_f):.3f}  feh>-1.2 占比={np.sum(w_f[feh_unl > -1.2])*100:.1f}%")
print(f"正样本           中位 feh={np.median(feh_pos):.3f}  feh>-1.2 占比={np.mean(feh_pos > -1.2)*100:.1f}%")


## 3. 运行 PU Bagging（T=500）

标准等概率 vs feh 匹配 vs feh KDE，三者同划分同种子。

In [ ]:
# 运行 PU Bagging：标准等概率 vs feh-only 匹配 vs feh-only KDE

T = 500

print("[base] 标准 PU 等概率采样 (T=500) ...")
te_base, p_base = imp.run_pu_bagging(X_tr, X_te, X_all, y_tr, y_te, n_pos_tr,
                                     imp.UniformSampler(unl_tr_idx), T, label="base", report_every=100)

print()
print("[feh_matched] feh-only 分箱匹配近邻 (T=500) ...")
te_fm, p_fm = imp.run_pu_bagging(X_tr, X_te, X_all, y_tr, y_te, n_pos_tr,
                                 fm_sampler, T, label="feh_matched", report_every=100)

print()
print("[feh_kde] feh-only KDE 加权 (T=500) ...")
te_fk, p_fk = imp.run_pu_bagging(X_tr, X_te, X_all, y_tr, y_te, n_pos_tr,
                                 fk_sampler, T, label="feh_kde", report_every=100)

res_base = imp.metrics(y_te, te_base)
res_fm = imp.metrics(y_te, te_fm)
res_fk = imp.metrics(y_te, te_fk)

print()
for nm, r in [("标准 PU(等概率)", res_base), ("feh 分箱匹配近邻", res_fm), ("feh KDE 加权", res_fk)]:
    print(f"{nm:20s} ROC={r['roc']:.4f}  PR={r['pr']:.4f}  P@50={r['p50']:.4f}  P@100={r['p100']:.4f}")


## 4. 三路对比（指标 + PR/ROC 曲线）

In [ ]:
# 三路对比：指标表 + PR/ROC 曲线

from sklearn.metrics import precision_recall_curve, roc_curve, auc

comp = pd.DataFrame({
    "采样方式": ["标准 PU (等概率)", "feh 分箱匹配近邻", "feh KDE 加权"],
    "ROC-AUC": [res_base["roc"], res_fm["roc"], res_fk["roc"]],
    "PR-AUC": [res_base["pr"], res_fm["pr"], res_fk["pr"]],
    "P@50": [res_base["p50"], res_fm["p50"], res_fk["p50"]],
    "P@100": [res_base["p100"], res_fm["p100"], res_fk["p100"]],
})
print(comp.to_string(index=False))

base_rate = y_te.sum() / len(y_te)
series = [("标准 PU", te_base, res_base, "b"), ("feh 匹配", te_fm, res_fm, "darkorange"), ("feh KDE", te_fk, res_fk, "green")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax1 = axes[0]
for nm, te, r, c in series:
    pr, rc, _ = precision_recall_curve(y_te, te)
    ax1.plot(rc, pr, color=c, linewidth=2, label=f"{nm} (AP={r['pr']:.3f})")
ax1.axhline(base_rate, color="gray", linestyle="--", linewidth=1.0, label=f"Random ({base_rate:.3f})")
ax1.set_xlabel("Recall"); ax1.set_ylabel("Precision")
ax1.set_title("Precision-Recall Curve")
ax1.legend(fontsize=9); ax1.grid(alpha=0.2); ax1.set_xlim(0, 1.02); ax1.set_ylim(0, 1.02)

ax2 = axes[1]
for nm, te, r, c in series:
    fpr, tpr, _ = roc_curve(y_te, te)
    ax2.plot(fpr, tpr, color=c, linewidth=2, label=f"{nm} (AUC={auc(fpr, tpr):.3f})")
ax2.plot([0, 1], [0, 1], "gray", linestyle="--", linewidth=1.0)
ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve")
ax2.legend(fontsize=9); ax2.grid(alpha=0.2); ax2.set_xlim(0, 1.02); ax2.set_ylim(0, 1.02)

fig.suptitle("标准 PU vs feh-only 重要性采样", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 5. 已知 CN 星标定阈值 + 候选体 [Fe/H] 分解

In [ ]:
# 已知 CN 星标定阈值 + 候选体 [Fe/H] 分解

recall_quantile = 0.05

def feh_split(prob, thr):
    cand = (stars_clean["label"].values == -1) & (prob >= thr)
    f = feh_all[cand]
    return dict(n=int(cand.sum()), gt=int((f > -1.2).sum()), le=int((f <= -1.2).sum()),
                median=float(np.median(f)))

thr_base, ncand_base, _ = imp.threshold_candidates(stars_clean, p_base, recall_quantile)
thr_fm, ncand_fm, _ = imp.threshold_candidates(stars_clean, p_fm, recall_quantile)
thr_fk, ncand_fk, _ = imp.threshold_candidates(stars_clean, p_fk, recall_quantile)
fs_base, fs_fm, fs_fk = feh_split(p_base, thr_base), feh_split(p_fm, thr_fm), feh_split(p_fk, thr_fk)

print("候选体 [Fe/H] 分解:")
for nm, thr, nc, fs in [("标准 PU(等概率)", thr_base, ncand_base, fs_base),
                        ("feh 分箱匹配近邻", thr_fm, ncand_fm, fs_fm),
                        ("feh KDE 加权", thr_fk, ncand_fk, fs_fk)]:
    print(f"  {nm:18s} thr={thr:.4f}  n={nc:5d}  feh>-1.2: {fs['gt']:5d}  feh<=-1.2: {fs['le']:5d}  中位feh={fs['median']:.3f}")

outpath = str(_PROJECT_ROOT / "XGB" / "XGB_PU_feholly_matched_candidates_threshold.csv")
imp.export_candidates(stars_clean, p_fm, thr_fm, outpath)
print(f"\nfeh 分箱匹配候选体已导出: {outpath}")


## 6. 五路对比（与 3D 重要性采样合并）

In [ ]:
# 五路对比：与 3D（teff/logg/feh）重要性采样合并

import json
fp = _PROJECT_ROOT / "XGB" / "importance_sampling_results.json"
if not fp.exists():
    print("未找到 importance_sampling_results.json（需先运行 3D 实验）。")
else:
    saved = json.loads(fp.read_text(encoding="utf-8"))["results"]
    def row(nm, r):
        return {"采样方式": nm, "ROC-AUC": f"{r['roc']:.4f}", "PR-AUC": f"{r['pr']:.4f}",
                "候选数": r["n_candidates"], "feh>-1.2": r["feh"]["n_feh_gt_-1.2"],
                "feh<=-1.2": r["feh"]["n_feh_le_-1.2"]}
    feh_fm = {"roc": res_fm["roc"], "pr": res_fm["pr"], "n_candidates": ncand_fm, "feh": fs_fm}
    feh_fk = {"roc": res_fk["roc"], "pr": res_fk["pr"], "n_candidates": ncand_fk, "feh": fs_fk}
    rows = [
        row("标准 PU (等概率)", saved["baseline"]),
        row("3D 分箱匹配近邻", saved["matched"]),
        row("feh 分箱匹配近邻", feh_fm),
        row("3D KDE 加权", saved["kde"]),
        row("feh KDE 加权", feh_fk),
    ]
    print(pd.DataFrame(rows).to_string(index=False))


## 7. 结论

**feh-only 重要性采样 vs 3D（teff/logg/feh）重要性采样：**

1. **是否只是 feh 的影响**：把 feh-only 匹配/KDE 与 3D 匹配/KDE 逐项对比（第 6 节）
   - feh-only ≈ 3D → [Fe/H] 是主导混淆变量
   - feh-only 明显更弱 → teff/logg 也贡献了混淆
2. **feh 分箱匹配 vs feh KDE 加权**：硬匹配通常比软加权对齐更激进，比较两者幅度即可判断"对齐强度"是否敏感
3. **与标准 PU 对比**：重要性采样（无论维度）都通过"抽掉物理捷径"暴露更弱的纯 CN 信号

> 判读要点：本实验不是要"改进"指标，而是要回答"基线高分里有多少来自 feh 这个混淆变量"。